# VPU 硬件测试 - 分步测试流程

本notebook将VPU测试分为以下步骤：
1. **BRAM读写测试** - 测试global_bram和inst_bram基本读写
2. **VPU寄存器测试** - 测试VPU_AXI_Regs寄存器访问
3. **INST_Decoder测试** - 测试指令解码器基本功能
4. **CDMA搬运测试** - 通过指令解码器控制CDMA
5. **VPU运算测试** - 测试VPU Max Pooling功能

## 地址映射（来自 rtl/vpu/vpu_defines.vh）
- `0x1000_0000` - HBM BRAM (1MB) - 外部数据暂存区（暂时替代HBM）
- `0x1010_0000` - inst_bram (128KB) - 指令存储区
- `0x1012_0000` - VPU GB (128KB) - VPU全局缓冲区
- `0x1014_0000` - VPU WB (32KB) - VPU权重缓冲区
- `0x1014_8000` - VPU_AXI_Regs (4KB) - VPU控制寄存器

In [1]:
# 导入依赖库
import sys
import time
import struct
import numpy as np
from pathlib import Path

# 添加测试模块路径
parent_dir = Path.cwd()
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from xdma_helpers import write_blob, read_blob, write_reg, read_reg
from xdma_helpers import GLOBAL_BRAM_BASE, INST_BRAM_BASE, VPU_GB_BASE, VPU_WB_BASE, VPU_REGS_BASE

print("✓ 模块导入成功")
print(f"地址映射 (来自 vpu_defines.vh):")
print(f"  HBM_BRAM:    0x{GLOBAL_BRAM_BASE:08X}")
print(f"  INST_BRAM:   0x{INST_BRAM_BASE:08X}")
print(f"  VPU_GB:      0x{VPU_GB_BASE:08X}")
print(f"  VPU_WB:      0x{VPU_WB_BASE:08X}")
print(f"  VPU_REGS:    0x{VPU_REGS_BASE:08X}")

✓ 模块导入成功
地址映射 (来自 vpu_defines.vh):
  HBM_BRAM:    0x10000000
  INST_BRAM:   0x10100000
  VPU_GB:      0x10120000
  VPU_WB:      0x10140000
  VPU_REGS:    0x10148000


## 步骤 1: BRAM 基本读写测试

测试 global_bram 和 inst_bram 的基本读写功能

In [2]:
# 测试 1.1: global_bram 读写 - 详细诊断版本
print("=" * 60)
print("测试 1.1: global_bram 读写 (256个fp32) - 详细诊断")
print("=" * 60)

# 生成256个fp32测试数据
num_elements = 256
test_data = np.arange(num_elements, dtype=np.float32)
print(f"写入数据大小: {len(test_data)} 个 fp32 ({len(test_data)*4} 字节)")
print(f"写入数据前8个: {test_data[:8]}")
print(f"写入数据后8个: {test_data[-8:]}")

# 写入数据
write_blob(GLOBAL_BRAM_BASE, test_data.tobytes())
print(f"✓ 数据已写入 0x{GLOBAL_BRAM_BASE:08X}")

# 读回数据
read_back = read_blob(GLOBAL_BRAM_BASE, len(test_data) * 4)
read_data = np.frombuffer(read_back, dtype=np.float32)

# 详细比较
print(f"\n【详细诊断】")
if np.array_equal(test_data, read_data):
    print(f"✓ global_bram 读写一致")
    print(f"  回读数据前8个: {read_data[:8]}")
    print(f"  回读数据后8个: {read_data[-8:]}")
else:
    print(f"✗ global_bram 读写不一致")
    print(f"  期望前8个: {test_data[:8]}")
    print(f"  实际前8个: {read_data[:8]}")
    
    # 找出所有不一致的位置
    diff_idx = np.where(test_data != read_data)[0]
    print(f"\n不一致统计:")
    print(f"  总数量: {len(diff_idx)}/{len(test_data)} ({100*len(diff_idx)/len(test_data):.1f}%)")
    
    if len(diff_idx) > 0:
        # 显示前10个不一致的位置
        print(f"\n前10个不一致位置:")
        for i in range(min(10, len(diff_idx))):
            idx = diff_idx[i]
            exp_val = test_data[idx]
            act_val = read_data[idx]
            exp_hex = struct.pack('f', exp_val).hex()
            act_hex = struct.pack('f', act_val).hex()
            print(f"  [{idx:3d}] 期望={exp_val:8.2f} (0x{exp_hex}), 实际={act_val:8.2f} (0x{act_hex})")
        
        # 检查是否有规律（例如每隔N个字节出错）
        if len(diff_idx) > 1:
            diffs = np.diff(diff_idx)
            unique_diffs = np.unique(diffs)
            print(f"\n不一致位置的间隔: {unique_diffs[:10]}")
            if len(unique_diffs) == 1:
                print(f"  → 固定间隔: 每隔 {unique_diffs[0]} 个元素")
        
        # 字节级分析：检查是否某个字节位置总是出错
        print(f"\n字节级分析 (前3个错误元素):")
        for i in range(min(3, len(diff_idx))):
            idx = diff_idx[i]
            exp_bytes = struct.pack('f', test_data[idx])
            act_bytes = struct.pack('f', read_data[idx])
            print(f"  元素[{idx}]:")
            print(f"    期望字节: {exp_bytes.hex()} = [{' '.join(f'{b:02x}' for b in exp_bytes)}]")
            print(f"    实际字节: {act_bytes.hex()} = [{' '.join(f'{b:02x}' for b in act_bytes)}]")
            for j in range(4):
                if exp_bytes[j] != act_bytes[j]:
                    print(f"      → 字节{j}不同: {exp_bytes[j]:02x} vs {act_bytes[j]:02x}")

测试 1.1: global_bram 读写 (256个fp32) - 详细诊断
写入数据大小: 256 个 fp32 (1024 字节)
写入数据前8个: [0. 1. 2. 3. 4. 5. 6. 7.]
写入数据后8个: [248. 249. 250. 251. 252. 253. 254. 255.]
✓ 数据已写入 0x10000000

【详细诊断】
✓ global_bram 读写一致
  回读数据前8个: [0. 1. 2. 3. 4. 5. 6. 7.]
  回读数据后8个: [248. 249. 250. 251. 252. 253. 254. 255.]


In [3]:
# 测试 1.2: inst_bram 读写 (fp32)
print("=" * 60)
print("测试 1.2: inst_bram 读写 (256个fp32)")
print("=" * 60)

# 生成256个fp32测试数据
test_data = np.arange(256, dtype=np.float32)
print(f"写入数据大小: {len(test_data)} 个 fp32 ({len(test_data)*4} 字节)")
print(f"写入数据前8个: {test_data[:8]}")

write_blob(INST_BRAM_BASE, test_data.tobytes())
read_back = read_blob(INST_BRAM_BASE, len(test_data) * 4)
read_data = np.frombuffer(read_back, dtype=np.float32)

if np.array_equal(test_data, read_data):
    print(f"✓ inst_bram 读写一致")
    print(f"  回读数据前8个: {read_data[:8]}")
    print(f"  回读数据后8个: {read_data[-8:]}")
else:
    print(f"✗ inst_bram 读写失败")
    print(f"  期望前8个: {test_data[:8]}")
    print(f"  实际前8个: {read_data[:8]}")
    # 找出不一致的位置
    diff_idx = np.where(test_data != read_data)[0]
    if len(diff_idx) > 0:
        print(f"  不一致数量: {len(diff_idx)}/{len(test_data)}")
        print(f"  首个不一致位置: {diff_idx[0]}, 期望={test_data[diff_idx[0]]}, 实际={read_data[diff_idx[0]]}")

测试 1.2: inst_bram 读写 (256个fp32)
写入数据大小: 256 个 fp32 (1024 字节)
写入数据前8个: [0. 1. 2. 3. 4. 5. 6. 7.]
✓ inst_bram 读写一致
  回读数据前8个: [0. 1. 2. 3. 4. 5. 6. 7.]
  回读数据后8个: [248. 249. 250. 251. 252. 253. 254. 255.]


## 步骤 2: VPU 寄存器测试

测试 VPU_AXI_Regs 寄存器的读写功能

In [4]:
# 定义VPU寄存器偏移（新架构）
# 可读写的寄存器：
REG_STATUS = 0x04          # [0] VPU ready (只读)
REG_DECODER_CTRL = 0x38    # [0] Decoder start (读写)
REG_INST_COUNT = 0x3C      # 指令数量 (读写)
REG_DECODER_STATUS = 0x40  # [0] busy, [1] done, [31] error (只读)

# 已废弃的寄存器（读取会返回0）：
# 0x08-0x34: UNIT_CHOOSE, mp_src_h, mp_src_w等VPU配置寄存器
# 这些现在由INST_Decoder的VPU_EXEC指令直接控制，不经过AXI寄存器

print("=" * 60)
print("测试 2: VPU 寄存器读写")
print("=" * 60)

# 读取VPU状态
status = read_reg(VPU_REGS_BASE, REG_STATUS)
print(f"VPU Status = 0x{status:08X}")
print(f"  [0] ready = {status & 0x1}")

# 读取解码器状态
decoder_status = read_reg(VPU_REGS_BASE, REG_DECODER_STATUS)
print(f"Decoder Status = 0x{decoder_status:08X}")
print(f"  [0] busy = {decoder_status & 0x1}")
print(f"  [1] done = {(decoder_status >> 1) & 0x1}")

# 测试可读写的INST_COUNT寄存器
print(f"\n测试解码器控制寄存器...")
write_reg(VPU_REGS_BASE, REG_INST_COUNT, 0x12345678)
inst_count_rb = read_reg(VPU_REGS_BASE, REG_INST_COUNT)
if inst_count_rb == 0x12345678:
    print(f"✓ INST_COUNT 寄存器读写成功 (0x{inst_count_rb:08X})")
else:
    print(f"✗ INST_COUNT 读写失败, 期望 0x12345678, 实际 0x{inst_count_rb:08X}")

# 架构说明
print(f"\n架构说明:")
print(f"  【新架构】软件 -> INST_Decoder -> VPU")
print(f"  - 软件只写 DECODER_CTRL/INST_COUNT 启动指令序列")
print(f"  - INST_Decoder 解析指令，直接控制 VPU 和 CDMA")
print(f"  - VPU配置寄存器(0x08-0x34)已从AXI寄存器接口移除")
print(f"  - 读取废弃地址会返回 0x00000000")
print(f"✓ 寄存器接口工作正常")

测试 2: VPU 寄存器读写
VPU Status = 0x00000001
  [0] ready = 1
Decoder Status = 0x00000002
  [0] busy = 0
  [1] done = 1

测试解码器控制寄存器...
✓ INST_COUNT 寄存器读写成功 (0x12345678)

架构说明:
  【新架构】软件 -> INST_Decoder -> VPU
  - 软件只写 DECODER_CTRL/INST_COUNT 启动指令序列
  - INST_Decoder 解析指令，直接控制 VPU 和 CDMA
  - VPU配置寄存器(0x08-0x34)已从AXI寄存器接口移除
  - 读取废弃地址会返回 0x00000000
✓ 寄存器接口工作正常


## 步骤 3: 指令解码器功能测试

定义指令编码函数和解码器控制函数

In [5]:
# 指令操作码定义
OP_NOP = 0x0
OP_CDMA_COPY = 0x1
OP_VPU_EXEC = 0x2
OP_WAIT_CDMA = 0x3
OP_WAIT_VPU = 0x4
OP_END = 0xF

# VPU 单元代码定义 (来自 Global_VPU.v)
UNIT_DQA = 1   # DeQuantize + Activation
UNIT_NN  = 2   # Neural Network LUT
UNIT_QA  = 3   # Quantize + Activation
UNIT_MP  = 4   # Max Pooling
UNIT_US  = 5   # UpSample
UNIT_AD  = 6   # Add (元素级加法)

print("VPU 单元代码:")
print(f"  UNIT_DQA = {UNIT_DQA} (DeQuantize)")
print(f"  UNIT_NN  = {UNIT_NN} (NN LUT)")
print(f"  UNIT_QA  = {UNIT_QA} (Quantize)")
print(f"  UNIT_MP  = {UNIT_MP} (Max Pooling)")
print(f"  UNIT_US  = {UNIT_US} (UpSample)")
print(f"  UNIT_AD  = {UNIT_AD} (Add)")

def _make_header(opcode, body_length, flags=0):
    """构造指令头: [31:28]OPCODE | [27:24]FLAGS | [23:0]BODY_LENGTH"""
    return ((opcode & 0xF) << 28) | ((flags & 0xF) << 24) | (body_length & 0xFFFFFF)

def encode_cdma_copy(src_addr, dst_addr, length):
    """CDMA_COPY指令: 4 words"""
    header = _make_header(OP_CDMA_COPY, 12)
    return struct.pack('<IIII', header, src_addr, dst_addr, length)

def encode_wait_cdma():
    """WAIT_CDMA指令: 1 word"""
    header = _make_header(OP_WAIT_CDMA, 0)
    return struct.pack('<I', header)

def encode_wait_vpu():
    """WAIT_VPU指令: 1 word"""
    header = _make_header(OP_WAIT_VPU, 0)
    return struct.pack('<I', header)

def encode_vpu_exec(unit_choose, src_addr, src2_addr, src_c, src_h, src_w,
                    bias_addr, scale_addr, dst_addr, addr_break, addr_s, addr_t):
    """VPU_EXEC指令: 13 words"""
    header = _make_header(OP_VPU_EXEC, 48)
    return struct.pack('<IIIIIIIIIIIII', header, unit_choose, src_addr, src2_addr,
                       src_c, src_h, src_w, bias_addr, scale_addr, dst_addr,
                       addr_break, addr_s, addr_t)

def encode_end():
    """END指令: 1 word"""
    header = _make_header(OP_END, 0)
    return struct.pack('<I', header)

def build_instruction_sequence(instructions):
    """合并指令列表"""
    return b''.join(instructions)

print("\n✓ 指令编码函数定义完成")

VPU 单元代码:
  UNIT_DQA = 1 (DeQuantize)
  UNIT_NN  = 2 (NN LUT)
  UNIT_QA  = 3 (Quantize)
  UNIT_MP  = 4 (Max Pooling)
  UNIT_US  = 5 (UpSample)
  UNIT_AD  = 6 (Add)

✓ 指令编码函数定义完成


In [6]:
# 解码器控制函数
def decoder_start(inst_count):
    """启动解码器"""
    write_reg(VPU_REGS_BASE, REG_DECODER_CTRL, 0x00)
    time.sleep(0.001)
    write_reg(VPU_REGS_BASE, REG_INST_COUNT, inst_count)
    write_reg(VPU_REGS_BASE, REG_DECODER_CTRL, 0x01)

def decoder_wait(timeout=5.0):
    """等待解码器完成"""
    deadline = time.time() + timeout
    seen_busy = False
    
    while time.time() < deadline:
        status = read_reg(VPU_REGS_BASE, REG_DECODER_STATUS)
        busy = status & 0x01
        done = (status >> 1) & 0x01
        error = (status >> 31) & 0x01
        
        if error:
            raise RuntimeError(f"Decoder error: status=0x{status:08X}")
        
        if busy:
            seen_busy = True
        
        if done and not busy:
            return status
        if seen_busy and status == 0:
            return status
        
        time.sleep(0.001)
    
    final_status = read_reg(VPU_REGS_BASE, REG_DECODER_STATUS)
    raise TimeoutError(f"Decoder timeout: status=0x{final_status:08X}")

print("✓ 解码器控制函数定义完成")

✓ 解码器控制函数定义完成


## 步骤 4: CDMA 搬运测试

通过指令解码器控制CDMA进行数据搬运

In [7]:
# 测试 4.1: CDMA 搬运 global_bram → VPU_GB (多位置多数据测试)
print("=" * 60)
print("测试 4.1: CDMA 搬运 (global_bram → VPU_GB) - 多位置多数据")
print("=" * 60)

# 定义多个测试用例，每个用例使用不同的位置和数据
test_cases = [
    {
        'name': '测试1: 低地址 uint8',
        'src_offset': 0x0000,
        'dst_offset': 0x0000,
        'size': 512,
        'dtype': np.uint8,
        'data_gen': lambda n: np.arange(n, dtype=np.uint8)
    },
    {
        'name': '测试2: 中地址 fp32',
        'src_offset': 0x2000,
        'dst_offset': 0x1000,
        'size': 256,  # 256字节 = 64个fp32
        'dtype': np.float32,
        'data_gen': lambda n: np.linspace(0, 100, n, dtype=np.float32)
    },
    {
        'name': '测试3: 高地址 int16',
        'src_offset': 0x8000,
        'dst_offset': 0x4000,
        'size': 1024,  # 1024字节 = 512个int16
        'dtype': np.int16,
        'data_gen': lambda n: (np.random.rand(n) * 1000 - 500).astype(np.int16)
    }
]

# 执行所有测试用例
for i, test in enumerate(test_cases, 1):
    print(f"\n{'='*50}")
    print(f"{test['name']}")
    print(f"{'='*50}")
    
    # 计算元素数量
    elem_size = np.dtype(test['dtype']).itemsize
    num_elements = test['size'] // elem_size
    
    # 生成测试数据
    np.random.seed(42 + i)  # 固定种子保证可重复
    test_data_raw = test['data_gen'](num_elements)
    test_data_bytes = test_data_raw.tobytes()
    
    print(f"源地址: global_bram+0x{test['src_offset']:04X} (0x{GLOBAL_BRAM_BASE+test['src_offset']:08X})")
    print(f"目标地址: VPU_GB+0x{test['dst_offset']:04X} (0x{VPU_GB_BASE+test['dst_offset']:08X})")
    print(f"数据大小: {test['size']} 字节 = {num_elements} 个 {test['dtype']}")
    print(f"数据前4个: {test_data_raw[:4]}")
    
    # 写入测试数据到 global_bram
    write_blob(GLOBAL_BRAM_BASE + test['src_offset'], test_data_bytes)
    print(f"✓ 写入数据到 global_bram")
    
    # 构建CDMA指令
    instructions = [
        encode_cdma_copy(
            GLOBAL_BRAM_BASE + test['src_offset'], 
            VPU_GB_BASE + test['dst_offset'], 
            test['size']
        ),
        encode_wait_cdma(),
        encode_end(),
    ]
    inst_bytes = build_instruction_sequence(instructions)
    inst_count = len(inst_bytes) // 4
    
    # 写入指令并启动解码器
    write_blob(INST_BRAM_BASE, inst_bytes)
    
    start_time = time.time()
    decoder_start(inst_count)
    status = decoder_wait(timeout=3.0)
    elapsed = time.time() - start_time
    
    print(f"✓ CDMA完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")
    
    # 验证结果
    result_bytes = read_blob(VPU_GB_BASE + test['dst_offset'], test['size'])
    result_data = np.frombuffer(result_bytes, dtype=test['dtype'])
    
    if np.array_equal(test_data_raw, result_data):
        print(f"✓ {test['name']} 搬运成功")
        print(f"  VPU_GB 前4个: {result_data[:4]}")
        print(f"  VPU_GB 后4个: {result_data[-4:]}")
    else:
        print(f"✗ {test['name']} 搬运失败")
        # 找出不一致的位置
        diff_mask = test_data_raw != result_data
        diff_idx = np.where(diff_mask)[0]
        print(f"  不一致数量: {len(diff_idx)}/{len(test_data_raw)}")
        
        if len(diff_idx) > 0:
            # 显示前几个不一致的位置（最多显示5个）
            show_count = min(5, len(diff_idx))
            print(f"\n  前 {show_count} 个不一致位置:")
            
            for j in range(show_count):
                idx = diff_idx[j]
                # 显示不一致位置前后各3个元素的上下文
                start_idx = max(0, idx - 3)
                end_idx = min(len(test_data_raw), idx + 4)
                
                print(f"\n    [{j+1}] index={idx}, 字节偏移=0x{idx*elem_size:04X}")
                print(f"        期望值: {test_data_raw[idx]}")
                print(f"        实际值: {result_data[idx]}")
                print(f"        上下文[{start_idx}:{end_idx}]:")
                print(f"          期望: {test_data_raw[start_idx:end_idx]}")
                print(f"          实际: {result_data[start_idx:end_idx]}")
                # 用箭头标记不一致的位置
                marker_pos = idx - start_idx
                print(f"          指示: {' '*marker_pos}^^^^")
            
            if len(diff_idx) > show_count:
                print(f"\n  ... 还有 {len(diff_idx)-show_count} 个不一致位置未显示")
            
            # 分析不一致位置的分布
            if len(diff_idx) > 1:
                intervals = np.diff(diff_idx)
                print(f"\n  不一致间隔统计: min={intervals.min()}, max={intervals.max()}, mean={intervals.mean():.1f}")

print(f"\n{'='*60}")
print(f"测试 4.1 完成 - 共 {len(test_cases)} 个用例")
print(f"{'='*60}")

测试 4.1: CDMA 搬运 (global_bram → VPU_GB) - 多位置多数据

测试1: 低地址 uint8
源地址: global_bram+0x0000 (0x10000000)
目标地址: VPU_GB+0x0000 (0x10120000)
数据大小: 512 字节 = 512 个 <class 'numpy.uint8'>
数据前4个: [0 1 2 3]
✓ 写入数据到 global_bram
✓ CDMA完成: status=0x00000002, 耗时 0.589s
✓ 测试1: 低地址 uint8 搬运成功
  VPU_GB 前4个: [0 1 2 3]
  VPU_GB 后4个: [252 253 254 255]

测试2: 中地址 fp32
源地址: global_bram+0x2000 (0x10002000)
目标地址: VPU_GB+0x1000 (0x10121000)
数据大小: 256 字节 = 64 个 <class 'numpy.float32'>
数据前4个: [0.        1.5873016 3.1746032 4.7619047]
✓ 写入数据到 global_bram
✓ CDMA完成: status=0x00000002, 耗时 0.594s
✓ 测试2: 中地址 fp32 搬运成功
  VPU_GB 前4个: [0.        1.5873016 3.1746032 4.7619047]
  VPU_GB 后4个: [ 95.2381   96.82539  98.4127  100.     ]

测试3: 高地址 int16
源地址: global_bram+0x8000 (0x10008000)
目标地址: VPU_GB+0x4000 (0x10124000)
数据大小: 1024 字节 = 512 个 <class 'numpy.int16'>
数据前4个: [ 489   49 -218 -422]
✓ 写入数据到 global_bram
✓ CDMA完成: status=0x00000002, 耗时 0.587s
✓ 测试3: 高地址 int16 搬运成功
  VPU_GB 前4个: [ 489   49 -218 -422]
  VPU_GB 后4个: [  94 -45

In [8]:
# 测试 4.2: CDMA 往返搬运 (global_bram → VPU_GB → global_bram) - fp32
print("=" * 60)
print("测试 4.2: CDMA 往返搬运 (256个fp32)")
print("=" * 60)

num_elements = 256
size = num_elements * 4  # 256个fp32 = 1024字节
src_off = 0x2000
dst_off = 0x3000
test_data = np.arange(num_elements, dtype=np.float32)

print(f"数据大小: {num_elements} 个 fp32 ({size} 字节)")
print(f"测试数据前8个: {test_data[:8]}")

# 准备数据
write_blob(GLOBAL_BRAM_BASE + src_off, test_data.tobytes())
# 用特殊值填充目标区域以验证确实被覆盖
clear_data = np.full(num_elements, -999.0, dtype=np.float32)
write_blob(GLOBAL_BRAM_BASE + dst_off, clear_data.tobytes())
print(f"✓ 源数据写入 0x{GLOBAL_BRAM_BASE+src_off:08X}")
print(f"✓ 目标区域清空 0x{GLOBAL_BRAM_BASE+dst_off:08X} (填充-999.0)")

# 构建往返指令
instructions = [
    encode_cdma_copy(GLOBAL_BRAM_BASE + src_off, VPU_GB_BASE, size),
    encode_wait_cdma(),
    encode_cdma_copy(VPU_GB_BASE, GLOBAL_BRAM_BASE + dst_off, size),
    encode_wait_cdma(),
    encode_end(),
]
inst_bytes = build_instruction_sequence(instructions)
inst_count = len(inst_bytes) // 4

print(f"\n指令序列: {inst_count} words")
print(f"Step 1: 0x{GLOBAL_BRAM_BASE+src_off:08X} → 0x{VPU_GB_BASE:08X} ({size}B)")
print(f"Step 2: 0x{VPU_GB_BASE:08X} → 0x{GLOBAL_BRAM_BASE+dst_off:08X} ({size}B)")

write_blob(INST_BRAM_BASE, inst_bytes)
start_time = time.time()
decoder_start(inst_count)
status = decoder_wait(timeout=5.0)
elapsed = time.time() - start_time

print(f"✓ 往返搬运完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")

# 验证结果
result = np.frombuffer(read_blob(GLOBAL_BRAM_BASE + dst_off, size), dtype=np.float32)
if np.array_equal(test_data, result):
    print(f"✓ 往返搬运成功, 数据一致")
    print(f"  结果前8个: {result[:8]}")
    print(f"  结果后8个: {result[-8:]}")
else:
    print(f"✗ 往返搬运失败")
    print(f"  期望前8个: {test_data[:8]}")
    print(f"  实际前8个: {result[:8]}")
    # 找出不一致的位置
    diff_idx = np.where(test_data != result)[0]
    if len(diff_idx) > 0:
        print(f"  不一致数量: {len(diff_idx)}/{len(test_data)}")
        print(f"  首个不一致位置: {diff_idx[0]}, 期望={test_data[diff_idx[0]]}, 实际={result[diff_idx[0]]}")

测试 4.2: CDMA 往返搬运 (256个fp32)
数据大小: 256 个 fp32 (1024 字节)
测试数据前8个: [0. 1. 2. 3. 4. 5. 6. 7.]
✓ 源数据写入 0x10002000
✓ 目标区域清空 0x10003000 (填充-999.0)

指令序列: 11 words
Step 1: 0x10002000 → 0x10120000 (1024B)
Step 2: 0x10120000 → 0x10003000 (1024B)
✓ 往返搬运完成: status=0x00000002, 耗时 0.587s
✓ 往返搬运成功, 数据一致
  结果前8个: [0. 1. 2. 3. 4. 5. 6. 7.]
  结果后8个: [248. 249. 250. 251. 252. 253. 254. 255.]


## 步骤 5: VPU Max Pooling 运算测试

测试VPU的Max Pooling功能（完整流程）

In [9]:
# 测试 5: VPU Max Pooling 完整流程
print("=" * 60)
print("测试 5: VPU Max Pooling 运算")
print("=" * 60)

# Max Pooling 参数（匹配 mp_unit_fixed.sv）
ACT_CHANNEL_NUM = 128  # 通道数（硬编码为128）
ACT_HEIGHT = 10        # 输入高度
ACT_WIDTH = 10         # 输入宽度
KERNEL_SIZE = 5        # 5x5 pooling
PAD = 2                # padding=2 (SAME padding)

# 输出尺寸（SAME padding: 输出=输入）
OUT_HEIGHT = ACT_HEIGHT   # 10
OUT_WIDTH = ACT_WIDTH     # 10

# 数据大小（FP32 = 4 bytes）
ACT_SIZE = ACT_CHANNEL_NUM * ACT_HEIGHT * ACT_WIDTH  # 12800 个 FP32
ACT_BYTES = ACT_SIZE * 4  # 51200 bytes

OUT_SIZE = ACT_CHANNEL_NUM * OUT_HEIGHT * OUT_WIDTH  # 12800 个 FP32
OUT_BYTES = OUT_SIZE * 4  # 51200 bytes

# VPU 参数 - ★★★ 修复：使用不同的 SRC/DST 地址 ★★★
UNIT_MP = 4
SRC_ADDR = 0x0000      # VPU GB 起始位置
DST_ADDR = 0xC800      # 51200 bytes (0xC800) 后，避免地址冲突

# mp_unit_fixed 不使用这些参数（已硬编码），但仍需传入
SRC_C = ACT_CHANNEL_NUM
SRC_H = OUT_HEIGHT
SRC_W = OUT_WIDTH

print(f"Max Pooling 参数 (FP32, 硬编码, 修复版):")
print(f"  输入: C={ACT_CHANNEL_NUM}, H={ACT_HEIGHT}, W={ACT_WIDTH}")
print(f"  输出: C={SRC_C}, H={SRC_H}, W={SRC_W} (SAME padding)")
print(f"  SRC_ADDR = 0x{SRC_ADDR:X} (VPU GB)")
print(f"  DST_ADDR = 0x{DST_ADDR:X} (VPU GB, +{DST_ADDR//1024}KB)")
print(f"  ★ 使用不同地址避免冲突")
print(f"  数据大小: {ACT_BYTES} bytes")
print(f"  数据类型: FP32 (float32)")
print(f"  Kernel: {KERNEL_SIZE}x{KERNEL_SIZE}, Pad: {PAD}")

测试 5: VPU Max Pooling 运算
Max Pooling 参数 (FP32, 硬编码, 简化配置):
  输入: C=128, H=10, W=10
  输出: C=128, H=10, W=10 (SAME padding)
  SRC_ADDR = 0x0
  DST_ADDR = 0x0 (覆盖输入)
  数据大小: 51200 bytes
  数据类型: FP32 (float32)
  Kernel: 5x5, Pad: 2


In [10]:
# 准备测试数据 (FP32, HWC 格式)
print("\n准备输入数据 (FP32, HWC 格式)...")

# 生成测试数据: CHW 格式 (C, H, W)
shape_chw = (ACT_CHANNEL_NUM, ACT_HEIGHT, ACT_WIDTH)
act_data_chw = np.arange(np.prod(shape_chw), dtype=np.float32).reshape(shape_chw)

# ★★★ 关键：转换为 HWC 格式 (H, W, C) ★★★
# RTL 期望 HWC 布局：地址 = (h * W + w) * C + c
act_data = np.transpose(act_data_chw, (1, 2, 0))  # CHW -> HWC

# 转换为字节
act_data_bytes = act_data.tobytes()

print(f"✓ 生成测试数据: {len(act_data_bytes)} bytes")
print(f"  原始形状 (CHW): {shape_chw}")
print(f"  转换后形状 (HWC): {act_data.shape}")
print(f"  数据类型: {act_data.dtype}")
print(f"  数据范围: [{act_data.min():.1f}, {act_data.max():.1f}]")
print(f"  前8个值 (HWC): {act_data.flatten()[:8]}")
print(f"  对应原始 CHW 索引: channel=0, h=0, w=0..7")

# 预先清零 SRC 和 DST 区域（因为使用不同地址）
dst_zeros = np.zeros(OUT_SIZE, dtype=np.float32)
dst_zero_bytes = dst_zeros.tobytes()
write_blob(VPU_GB_BASE + SRC_ADDR, dst_zero_bytes)
write_blob(VPU_GB_BASE + DST_ADDR, dst_zero_bytes)
print(f"\n✓ VPU GB 区域已预清零")
print(f"  SRC: 0x{VPU_GB_BASE+SRC_ADDR:08X} (0x{SRC_ADDR:X}, {ACT_BYTES} bytes)")
print(f"  DST: 0x{VPU_GB_BASE+DST_ADDR:08X} (0x{DST_ADDR:X}, {OUT_BYTES} bytes)")

# 写入输入数据到 global_bram
write_blob(GLOBAL_BRAM_BASE + SRC_ADDR, act_data_bytes)
print(f"\n✓ 输入数据已写入 global_bram (0x{GLOBAL_BRAM_BASE+SRC_ADDR:08X})")


准备输入数据 (FP32, HWC 格式)...
✓ 生成测试数据: 51200 bytes
  原始形状 (CHW): (128, 10, 10)
  转换后形状 (HWC): (10, 10, 128)
  数据类型: float32
  数据范围: [0.0, 12799.0]
  前8个值 (HWC): [  0. 100. 200. 300. 400. 500. 600. 700.]
  对应原始 CHW 索引: channel=0, h=0, w=0..7

✓ VPU GB DST 区域已预写 0
  VPU GB dst: 0x10120000 (dst_addr=0x0, 51200 bytes)
✓ 输入数据已写入 global_bram (0x10000000)


In [11]:
# 构建 VPU Max Pooling 指令序列
print("\n构建指令序列...")

instructions = [
    # Step 1: CDMA 将数据搬到 VPU GB SRC 位置
    encode_cdma_copy(GLOBAL_BRAM_BASE + SRC_ADDR, VPU_GB_BASE + SRC_ADDR, ACT_BYTES),
    encode_wait_cdma(),
    
    # Step 2: 执行 VPU Max Pooling（SRC != DST，避免地址冲突）
    encode_vpu_exec(
        unit_choose=UNIT_MP,
        src_addr=SRC_ADDR,
        src2_addr=0,
        src_c=SRC_C,
        src_h=SRC_H,
        src_w=SRC_W,
        bias_addr=0,
        scale_addr=0,
        dst_addr=DST_ADDR,  # 使用不同的地址
        addr_break=0,
        addr_s=0,
        addr_t=0
    ),
    encode_wait_vpu(),
    
    # Step 3: CDMA 将结果从 DST 位置搬回 global_bram
    encode_cdma_copy(VPU_GB_BASE + DST_ADDR, GLOBAL_BRAM_BASE + DST_ADDR, OUT_BYTES),
    encode_wait_cdma(),
    
    encode_end(),
]

inst_bytes = build_instruction_sequence(instructions)
inst_count = len(inst_bytes) // 4

print(f"✓ 指令序列: {inst_count} words ({len(inst_bytes)} bytes)")
print(f"  CDMA IN:  0x{GLOBAL_BRAM_BASE+SRC_ADDR:08X} → 0x{VPU_GB_BASE+SRC_ADDR:08X} ({ACT_BYTES}B)")
print(f"  VPU:      Max Pooling src=0x{SRC_ADDR:X}, dst=0x{DST_ADDR:X} (不同地址)")
print(f"  CDMA OUT: 0x{VPU_GB_BASE+DST_ADDR:08X} → 0x{GLOBAL_BRAM_BASE+DST_ADDR:08X} ({OUT_BYTES}B)")


构建指令序列...
✓ 指令序列: 25 words (100 bytes)
  CDMA IN:  0x10000000 → 0x10120000 (51200B)
  VPU:      Max Pooling src=0x0, dst=0x0
  CDMA OUT: 0x10120000 → 0x10000000 (51200B)


In [12]:
# 执行 VPU Max Pooling
print("\n执行 VPU 运算...")

# 【诊断】执行前检查 VPU GB 的输入区域
print("\n【诊断】执行前检查:")
vpu_gb_before = read_blob(VPU_GB_BASE + SRC_ADDR, min(128, ACT_BYTES))
vpu_gb_before_fp32 = np.frombuffer(vpu_gb_before, dtype=np.float32)
print(f"  VPU GB SRC 区域前8个值: {vpu_gb_before_fp32[:8]}")
print(f"  (应该全是0，因为还没 CDMA 搬入)")

# 【诊断】检查 VPU 寄存器配置
print("\n【诊断】VPU 寄存器配置检查:")
vpu_config_ready = read_reg(VPU_REGS_BASE, 0x00)  # config_ready
print(f"  config_ready: 0x{vpu_config_ready:08X} (期望: 0x01, 所有单元 ready)")

# 写入指令
write_blob(INST_BRAM_BASE, inst_bytes)
print("\n✓ 指令序列已写入 inst_bram")

# 启动解码器
start_time = time.time()
decoder_start(inst_count)
print("✓ 解码器已启动")

# 等待完成
status = decoder_wait(timeout=10.0)
elapsed = time.time() - start_time

print(f"✓ VPU 运算完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")

# 【诊断】执行后检查 VPU GB 的输出区域（在 CDMA 搬出前）
print("\n【诊断】执行后检查:")
vpu_gb_after_dst = read_blob(VPU_GB_BASE + DST_ADDR, min(128, OUT_BYTES))
vpu_gb_after_dst_fp32 = np.frombuffer(vpu_gb_after_dst, dtype=np.float32)
print(f"  VPU GB DST 区域前8个值: {vpu_gb_after_dst_fp32[:8]}")
print(f"  (这是 VPU 计算后的结果，还在 VPU GB 里)")

# 【诊断】比较 VPU 输出和期望输出的第一个值
print(f"\n【关键诊断】第一个值对比:")
print(f"  VPU 输出 [0]: {vpu_gb_after_dst_fp32[0]}")
print(f"  期望输出 [0]: 22.0 (来自 5x5 窗口内 max(0,1,2,10,11,12,20,21,22))")
if abs(vpu_gb_after_dst_fp32[0] - 22.0) < 0.1:
    print(f"  ✓ 匹配！Max Pooling 工作正常")
elif vpu_gb_after_dst_fp32[0] == 0.0:
    print(f"  ✗ 输出是 0 - VPU 可能没有执行或输入数据有问题")
else:
    print(f"  ✗ 不匹配且非0 - VPU 执行了但结果错误")


执行 VPU 运算...

【诊断】执行前检查:
  VPU GB SRC 区域前8个值: [0. 0. 0. 0. 0. 0. 0. 0.]
  (应该全是0，因为还没 CDMA 搬入)

【诊断】VPU 寄存器配置检查:


TypeError: read_reg() missing 1 required positional argument: 'offset'

In [ ]:
# 调试：检查 VPU 输出的原始数据
print("\n" + "=" * 60)
print("调试信息 - VPU 输出数据检查")
print("=" * 60)

print("\n1. VPU 输出数据统计:")
result_bytes_raw = read_blob(GLOBAL_BRAM_BASE + DST_ADDR, OUT_BYTES)
result_fp32_raw = np.frombuffer(result_bytes_raw, dtype=np.float32)

print(f"  数据量: {len(result_fp32_raw)} FP32")
print(f"  数据范围: [{result_fp32_raw.min():.2f}, {result_fp32_raw.max():.2f}]")
print(f"  前32个值:")
for i in range(0, 32, 8):
    print(f"    [{i:2d}-{i+7:2d}]: {result_fp32_raw[i:i+8]}")

print(f"\n2. 与预清零值比较:")
print(f"  全零数量: {np.sum(result_fp32_raw == 0)}/{len(result_fp32_raw)}")
if np.all(result_fp32_raw == 0):
    print(f"  ⚠️ 所有输出都是0 - VPU 可能没有执行或没有写入结果!")
elif np.sum(result_fp32_raw == 0) > len(result_fp32_raw) * 0.8:
    print(f"  ⚠️ 大部分输出是0 - VPU 可能只处理了部分数据!")
else:
    print(f"  ✓ 输出有非零值，VPU 确实写入了数据")

print(f"\n3. 与输入数据比较:")
# 检查输出是否就是输入数据（说明没有做 pooling）
input_data_flat = act_data.flatten()
print(f"  输入数据前8个: {input_data_flat[:8]}")
print(f"  输出数据前8个: {result_fp32_raw[:8]}")
if np.array_equal(result_fp32_raw, input_data_flat):
    print(f"  ⚠️ 输出 == 输入 - VPU 直接复制了输入，没有做 Max Pooling!")
else:
    print(f"  ✓ 输出 != 输入")

print(f"\n4. 数值规律分析:")
print(f"  输出值是否都是 100 的倍数? {np.all(result_fp32_raw[result_fp32_raw != 0] % 100 == 0)}")
print(f"  前20个非零值: {result_fp32_raw[result_fp32_raw != 0][:20]}")

print("\n" + "=" * 60)

In [ ]:
# 读取并验证结果 (FP32)
print("\n读取结果...")

result_bytes = read_blob(GLOBAL_BRAM_BASE + DST_ADDR, OUT_BYTES)
result_fp32 = np.frombuffer(result_bytes, dtype=np.float32)

print(f"✓ 结果数据: {len(result_bytes)} bytes ({len(result_fp32)} FP32)")
print(f"  前8个值: {result_fp32[:8]}")
print(f"  数据范围: [{np.min(result_fp32):.1f}, {np.max(result_fp32):.1f}]")
print(f"  非零元素: {np.count_nonzero(result_fp32)}/{len(result_fp32)}")

In [ ]:
# 调试：检查实际写入和读取的数据
print("\n" + "=" * 60)
print("调试信息 - 数据验证")
print("=" * 60)

# 1. 检查写入 global_bram 的数据是否正确
print("\n1. 验证 global_bram 中的输入数据:")
verify_input = read_blob(GLOBAL_BRAM_BASE + SRC_ADDR, min(128, ACT_BYTES))  # 读取前128字节
verify_input_fp32 = np.frombuffer(verify_input, dtype=np.float32)
print(f"  global_bram 前8个值: {verify_input_fp32[:8]}")
print(f"  act_data 前8个值:    {act_data.flatten()[:8]}")
match = np.array_equal(verify_input_fp32[:8], act_data.flatten()[:8])
print(f"  匹配: {match}")

# 2. 检查 VPU GB 输入区域（执行前应该是0，执行后是输入数据）
print("\n2. 验证 VPU GB 输入区域（CDMA 搬入后）:")
# 注意：这个检查需要在 CDMA 搬运完成后进行
# 暂时跳过，因为我们已经执行完了

# 3. 分析期望值和实际值的规律
print("\n3. 分析数据规律:")
print(f"  act_data shape (HWC): {act_data.shape}")
print(f"  act_data flatten 前32个值:")
act_flat = act_data.flatten()
for i in range(0, 32, 8):
    print(f"    [{i:2d}-{i+7:2d}]: {act_flat[i:i+8]}")

# 4. 检查 Golden Model 使用的输入数据
print("\n4. Golden Model 输入检查:")
print(f"  Golden Model 使用的 act_data 前8个值: {act_data.flatten()[:8]}")
print(f"  这些值对应的位置:")
for i in range(8):
    h = i // (ACT_WIDTH * ACT_CHANNEL_NUM)
    w = (i // ACT_CHANNEL_NUM) % ACT_WIDTH
    c = i % ACT_CHANNEL_NUM
    print(f"    索引{i}: (h={h}, w={w}, c={c}) = {act_data[h, w, c]}")

# 5. 模拟 Max Pooling 的第一个输出（手动计算）
print("\n5. 手动计算位置(0,0,0)的 Max Pooling 结果:")
print(f"  Kernel size: 5x5, Padding: 2")
print(f"  对于 (h=0, w=0, c=0)，窗口范围: h∈[-2,2], w∈[-2,2]")
print(f"  考虑 padding 后，有效范围: h∈[0,2], w∈[0,2]")
window_vals = []
for h in range(0, min(3, ACT_HEIGHT)):
    for w in range(0, min(3, ACT_WIDTH)):
        val = act_data[h, w, 0]
        window_vals.append(val)
        print(f"    act_data[{h},{w},0] = {val}")
print(f"  窗口内所有值: {window_vals}")
print(f"  Max = {max(window_vals)}")
print(f"  但期望输出显示为: 22.0")
print(f"  实际 VPU 输出为: 0.0")

print("\n" + "=" * 60)

### Max Pooling 测试 - 修复版（使用不同的 SRC/DST 地址）

**问题**：之前的测试使用 `SRC_ADDR = DST_ADDR = 0`，导致 VPU 原地修改数据，可能与 CDMA 操作冲突。

**修复方案**：使用不同的 SRC 和 DST 地址，避免地址冲突。

In [ ]:
# Max Pooling 修复测试 - 使用不同的 SRC/DST 地址
print("=" * 60)
print("Max Pooling 修复测试（SRC != DST）")
print("=" * 60)

# 使用不同的地址
FIX_SRC_ADDR = 0x0000  # VPU GB 起始位置
FIX_DST_ADDR = 0xC800  # 51200 bytes (0xC800) 后，避免重叠

print(f"\n地址配置:")
print(f"  SRC_ADDR: 0x{FIX_SRC_ADDR:08X} (VPU GB)")
print(f"  DST_ADDR: 0x{FIX_DST_ADDR:08X} (VPU GB)")
print(f"  地址间隔: {FIX_DST_ADDR} bytes = {FIX_DST_ADDR // 1024} KB")
print(f"  避免重叠: ✓")

# 清零两个区域
print(f"\n准备数据...")
dst_zeros = np.zeros(OUT_SIZE, dtype=np.float32)
write_blob(VPU_GB_BASE + FIX_SRC_ADDR, dst_zeros.tobytes())
write_blob(VPU_GB_BASE + FIX_DST_ADDR, dst_zeros.tobytes())
print(f"✓ VPU GB SRC/DST 区域已清零")

# 写入输入数据到 global_bram
write_blob(GLOBAL_BRAM_BASE + SRC_ADDR, act_data_bytes)
print(f"✓ 输入数据已写入 global_bram")

# 构建指令序列（使用新地址）
instructions_fix = [
    # Step 1: CDMA 将数据搬到 VPU GB SRC 位置
    encode_cdma_copy(GLOBAL_BRAM_BASE + SRC_ADDR, VPU_GB_BASE + FIX_SRC_ADDR, ACT_BYTES),
    encode_wait_cdma(),
    
    # Step 2: 执行 VPU Max Pooling（SRC != DST）
    encode_vpu_exec(
        unit_choose=UNIT_MP,
        src_addr=FIX_SRC_ADDR,
        src2_addr=0,
        src_c=SRC_C,
        src_h=SRC_H,
        src_w=SRC_W,
        bias_addr=0,
        scale_addr=0,
        dst_addr=FIX_DST_ADDR,
        addr_break=0,
        addr_s=0,
        addr_t=0
    ),
    encode_wait_vpu(),
    
    # Step 3: CDMA 将结果从 DST 位置搬回 global_bram
    encode_cdma_copy(VPU_GB_BASE + FIX_DST_ADDR, GLOBAL_BRAM_BASE + DST_ADDR, OUT_BYTES),
    encode_wait_cdma(),
    
    encode_end(),
]

inst_bytes_fix = build_instruction_sequence(instructions_fix)
inst_count_fix = len(inst_bytes_fix) // 4

print(f"\n✓ 指令序列: {inst_count_fix} words")
print(f"  CDMA IN:  global_bram → VPU_GB[0x{FIX_SRC_ADDR:X}]")
print(f"  VPU:      Max Pooling [0x{FIX_SRC_ADDR:X}] → [0x{FIX_DST_ADDR:X}]")
print(f"  CDMA OUT: VPU_GB[0x{FIX_DST_ADDR:X}] → global_bram")

# 执行
write_blob(INST_BRAM_BASE, inst_bytes_fix)
start_time = time.time()
decoder_start(inst_count_fix)
status = decoder_wait(timeout=10.0)
elapsed = time.time() - start_time

print(f"\n✓ 执行完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")

# 检查 VPU GB 中的 SRC 和 DST
print(f"\n检查 VPU GB 数据:")
vpu_src_check = read_blob(VPU_GB_BASE + FIX_SRC_ADDR, 32)
vpu_src_fp32 = np.frombuffer(vpu_src_check, dtype=np.float32)
print(f"  SRC[0x{FIX_SRC_ADDR:X}] 前8个: {vpu_src_fp32}")

vpu_dst_check = read_blob(VPU_GB_BASE + FIX_DST_ADDR, 32)
vpu_dst_fp32 = np.frombuffer(vpu_dst_check, dtype=np.float32)
print(f"  DST[0x{FIX_DST_ADDR:X}] 前8个: {vpu_dst_fp32}")

# 读取 global_bram 结果
result_fix_bytes = read_blob(GLOBAL_BRAM_BASE + DST_ADDR, OUT_BYTES)
result_fix = np.frombuffer(result_fix_bytes, dtype=np.float32)

print(f"\nglobal_bram 输出前8个: {result_fix[:8]}")
print(f"\n关键对比:")
print(f"  实际输出[0]: {result_fix[0]}")
print(f"  期望输出[0]: 22.0")
print(f"  匹配: {abs(result_fix[0] - 22.0) < 0.1}")

if abs(result_fix[0] - 22.0) < 0.1:
    print(f"\n✓ 修复成功！使用不同的 SRC/DST 地址后，Max Pooling 工作正常")
else:
    print(f"\n✗ 仍然失败，问题可能在 VPU 硬件实现本身")

In [ ]:
# 计算 Golden Model 进行数值比较 (FP32, SAME padding, HWC 格式)
print("\n计算 Golden Model...")

def compute_max_pooling_golden_hwc(input_data_hwc, kernel_size=5, pad=2):
    """
    计算 Max Pooling 的 Golden Model (HWC 格式输入)
    input_data_hwc: shape (H, W, C)
    pad: padding 大小
    返回: shape (H, W, C) (SAME padding时输入输出尺寸相同)
    """
    H, W, C = input_data_hwc.shape
    
    # 应用 padding (用 -inf 填充)
    if pad > 0:
        padded = np.full((H + 2*pad, W + 2*pad, C), -np.inf, dtype=input_data_hwc.dtype)
        padded[pad:pad+H, pad:pad+W, :] = input_data_hwc
    else:
        padded = input_data_hwc
    
    out_h = H  # SAME padding
    out_w = W
    output = np.zeros((out_h, out_w, C), dtype=input_data_hwc.dtype)
    
    for h in range(out_h):
        for w in range(out_w):
            for c in range(C):
                # 提取 kernel 窗口
                window = padded[h:h+kernel_size, w:w+kernel_size, c]
                output[h, w, c] = np.max(window)
    
    return output

# 计算期望结果（输入已经是 HWC 格式）
expected = compute_max_pooling_golden_hwc(act_data, kernel_size=KERNEL_SIZE, pad=PAD)
expected_flat = expected.flatten()

print(f"✓ Golden Model 计算完成")
print(f"  输入形状 (HWC): {act_data.shape}")
print(f"  期望输出形状 (HWC): {expected.shape}")
print(f"  期望数据范围: [{expected.min():.1f}, {expected.max():.1f}]")
print(f"  期望元素数: {expected.size}")
print(f"  实际读取元素数: {len(result_fp32)}")

# 比较结果
if len(result_fp32) >= expected.size:
    result_reshaped = result_fp32[:expected.size].reshape(expected.shape)
    
    diff = np.abs(result_reshaped - expected)
    max_diff = np.max(diff)
    mean_diff = np.mean(diff)
    
    # 计算匹配率
    match_count = np.sum(diff < 0.1)
    match_rate = match_count / expected.size * 100
    
    print(f"\n数值比较:")
    print(f"  最大误差: {max_diff:.4f}")
    print(f"  平均误差: {mean_diff:.4f}")
    print(f"  匹配率: {match_rate:.2f}% ({match_count}/{expected.size})")
    
    # 显示详细对比（前64个值，每行8个）
    print(f"\n详细对比 (前64个值):")
    print("索引 | 期望值  | 实际值  | 误差")
    print("-" * 45)
    num_show = min(64, len(expected_flat))
    for i in range(num_show):
        err = abs(expected_flat[i] - result_fp32[i])
        status = "✓" if err < 0.1 else "✗"
        print(f"{i:3d}  | {expected_flat[i]:7.1f} | {result_fp32[i]:7.1f} | {err:6.2f} {status}")
        if (i + 1) % 16 == 0:
            print()  # 每16个值空一行
    
    # 显示误差分布
    print(f"\n误差分布:")
    print(f"  误差 < 0.01: {np.sum(diff < 0.01)} 个")
    print(f"  误差 < 0.1:  {np.sum(diff < 0.1)} 个")
    print(f"  误差 < 1.0:  {np.sum(diff < 1.0)} 个")
    print(f"  误差 >= 1.0: {np.sum(diff >= 1.0)} 个")
    
    # 找出误差最大的几个位置
    worst_indices = np.argsort(diff.flatten())[-5:][::-1]
    print(f"\n误差最大的5个位置:")
    for idx in worst_indices:
        # HWC 格式: idx = h * (W * C) + w * C + c
        H_out, W_out, C_out = expected.shape
        h = idx // (W_out * C_out)
        w = (idx // C_out) % W_out
        c = idx % C_out
        print(f"  [{idx}] (h={h},w={w},c={c}) 期望={expected_flat[idx]:.2f}, 实际={result_fp32[idx]:.2f}, 误差={diff.flatten()[idx]:.2f}")
    
    if match_rate > 95:
        print(f"\n✓ Max Pooling 数值验证通过!")
    else:
        print(f"\n⚠ Max Pooling 匹配率偏低，需要检查")
else:
    print(f"\n⚠ 数据量不匹配: 期望 {expected.size}, 实际 {len(result_fp32)}")

## 步骤 6: VPU ADD 运算测试

测试 VPU 的 ADD 单元（元素级加法）功能。

**测试说明**：
- ADD 单元执行元素级加法：`DST = SRC1 + SRC2`
- `UNIT_AD = 6`（对应 `ad_unit.sv`）
- `FP_CORE_NUM = 8`：每次处理 8 个 FP32
- 数据尺寸必须是 8 的倍数
- 使用 FP32 数据类型

**测试用例**：
1. 最小测试：C=8, H=1, W=1（1 个 BRAM word）
2. 多通道测试：C=16, H=1, W=1（2 个 BRAM words）
3. 2D 测试：C=8, H=2, W=2（4 个 BRAM words）
4. 大尺寸测试：C=128, H=4, W=4（256 个 BRAM words）

In [ ]:
# 测试 6: VPU ADD 功能（简化配置）
print("=" * 60)
print("测试 6: VPU ADD 运算（最小测试）")
print("=" * 60)

# ★★★ 最小测试：只测试 1 个 BRAM word = 8 个 FP32 ★★★
UNIT_ADD = 6

# 最小数据尺寸（正好 1 个 BRAM word）
ADD_C = 8     # 1 channel
ADD_H = 1     # 1 height  
ADD_W = 1     # 8 width = 8 FP32 = 1 BRAM word
ADD_SIZE = ADD_C * ADD_H * ADD_W  # 8 个 FP32
ADD_BYTES = ADD_SIZE * 4  # 32 bytes = 1 BRAM word

# VPU GB 内地址（字节地址）
ADD_SRC1_ADDR = 0x0000
ADD_SRC2_ADDR = 0x0020  # 32 bytes 后（1 word）
ADD_DST_ADDR = 0x0040   # 64 bytes 后（2 words）

# global_bram 地址
ADD_SRC1_OFF = 0x0
ADD_SRC2_OFF = 0x100
ADD_DST_OFF = 0x200

print(f"ADD 参数 (FP32, 最小测试):")
print(f"  UNIT_CHOOSE: {UNIT_ADD} (UNIT_AD)")
print(f"  FP_CORE_NUM: 8 (每次处理 8 个 FP32)")
print(f"  输入形状: C={ADD_C}, H={ADD_H}, W={ADD_W}")
print(f"  数据大小: {ADD_BYTES} bytes ({ADD_SIZE} FP32) = 1 BRAM word")
print(f"  数据类型: FP32 (float32)")
print(f"  运算: SRC1 + SRC2 = DST")

In [ ]:
# 准备ADD测试数据 (FP32, 最小测试)
print("\n准备测试数据 (FP32, 8个元素)...")

# 简单数据：1+10=11, 2+20=22, ..., 8+80=88
src1_data = np.arange(1, 9, dtype=np.float32)  # [1, 2, 3, 4, 5, 6, 7, 8]
src2_data = np.arange(10, 90, 10, dtype=np.float32)  # [10, 20, 30, 40, 50, 60, 70, 80]
expected_add = src1_data + src2_data  # [11, 22, 33, 44, 55, 66, 77, 88]

print(f"✓ SRC1 数据: {src1_data}")
print(f"✓ SRC2 数据: {src2_data}")
print(f"✓ 期望结果: {expected_add}")

# 写入数据到 global_bram
write_blob(GLOBAL_BRAM_BASE + ADD_SRC1_OFF, src1_data.tobytes())
write_blob(GLOBAL_BRAM_BASE + ADD_SRC2_OFF, src2_data.tobytes())

print(f"\n✓ 数据已写入 global_bram")
print(f"  SRC1: 0x{GLOBAL_BRAM_BASE+ADD_SRC1_OFF:08X} ({len(src1_data.tobytes())} bytes)")
print(f"  SRC2: 0x{GLOBAL_BRAM_BASE+ADD_SRC2_OFF:08X} ({len(src2_data.tobytes())} bytes)")

# 预先将 dst_addr 对应范围清零，再执行 add_test
dst_zeros = np.zeros(ADD_SIZE, dtype=np.float32)
dst_zero_bytes = dst_zeros.tobytes()
write_blob(VPU_GB_BASE + ADD_DST_ADDR, dst_zero_bytes)
write_blob(GLOBAL_BRAM_BASE + ADD_DST_OFF, dst_zero_bytes)
print(f"\n✓ DST 区域已预写 0")
print(f"  VPU GB dst: 0x{VPU_GB_BASE+ADD_DST_ADDR:08X} (dst_addr=0x{ADD_DST_ADDR:04X}, {ADD_BYTES} bytes)")
print(f"  global_bram: 0x{GLOBAL_BRAM_BASE+ADD_DST_OFF:08X} ({ADD_BYTES} bytes)")

In [ ]:
# 构建 VPU ADD 指令序列
print("\n构建指令序列...")

instructions = [
    # Step 1: CDMA 将 SRC1 搬到 VPU GB
    encode_cdma_copy(GLOBAL_BRAM_BASE + ADD_SRC1_OFF, VPU_GB_BASE + ADD_SRC1_ADDR, ADD_BYTES),
    encode_wait_cdma(),
    
    # Step 2: CDMA 将 SRC2 搬到 VPU GB
    encode_cdma_copy(GLOBAL_BRAM_BASE + ADD_SRC2_OFF, VPU_GB_BASE + ADD_SRC2_ADDR, ADD_BYTES),
    encode_wait_cdma(),
    
    # Step 3: 执行 VPU ADD
    encode_vpu_exec(
        unit_choose=UNIT_ADD,
        src_addr=ADD_SRC1_ADDR,  # VPU GB 内部地址
        src2_addr=ADD_SRC2_ADDR,  # 第二个操作数地址
        src_c=ADD_C,
        src_h=ADD_H,
        src_w=ADD_W,
        bias_addr=0,
        scale_addr=0,
        dst_addr=ADD_DST_ADDR,
        addr_break=0,
        addr_s=0,
        addr_t=0
    ),
    encode_wait_vpu(),
    
    # Step 4: CDMA 将结果搬回 global_bram
    encode_cdma_copy(VPU_GB_BASE + ADD_DST_ADDR, GLOBAL_BRAM_BASE + ADD_DST_OFF, ADD_BYTES),
    encode_wait_cdma(),
    
    encode_end(),
]

inst_bytes = build_instruction_sequence(instructions)
inst_count = len(inst_bytes) // 4

print(f"✓ 指令序列: {inst_count} words ({len(inst_bytes)} bytes)")
print(f"  CDMA IN SRC1: 0x{GLOBAL_BRAM_BASE+ADD_SRC1_OFF:08X} → 0x{VPU_GB_BASE+ADD_SRC1_ADDR:08X}")
print(f"  CDMA IN SRC2: 0x{GLOBAL_BRAM_BASE+ADD_SRC2_OFF:08X} → 0x{VPU_GB_BASE+ADD_SRC2_ADDR:08X}")
print(f"  VPU:          ADD (C={ADD_C}, H={ADD_H}, W={ADD_W})")
print(f"  CDMA OUT:     0x{VPU_GB_BASE+ADD_DST_ADDR:08X} → 0x{GLOBAL_BRAM_BASE+ADD_DST_OFF:08X}")

In [ ]:
# 执行 VPU ADD
print("\n执行 VPU 运算...")

# 写入指令
write_blob(INST_BRAM_BASE, inst_bytes)
print("✓ 指令序列已写入 inst_bram")

# 启动解码器
start_time = time.time()
decoder_start(inst_count)
print("✓ 解码器已启动")

# 等待完成
status = decoder_wait(timeout=10.0)
elapsed = time.time() - start_time

print(f"✓ VPU ADD 完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")

In [ ]:
# 读取并验证 ADD 结果 (FP32, 最小测试)
print("\n读取结果...")

result_bytes = read_blob(GLOBAL_BRAM_BASE + ADD_DST_OFF, ADD_BYTES)
result_add = np.frombuffer(result_bytes, dtype=np.float32)

print(f"✓ 结果数据: {len(result_bytes)} bytes ({len(result_add)} FP32)")
print(f"  实际输出: {result_add}")
print(f"  期望输出: {expected_add}")

# 数值比较
diff = np.abs(result_add - expected_add)
max_diff = np.max(diff)

# 详细对比
print(f"\n详细对比:")
print("索引 | SRC1  | SRC2  | 期望  | 实际  | 误差  ")
print("-" * 55)
for i in range(len(expected_add)):
    err = diff[i]
    status = "✓" if err < 0.1 else "✗"
    print(f"{i:3d}  | {src1_data[i]:5.1f} | {src2_data[i]:5.1f} | {expected_add[i]:5.1f} | {result_add[i]:5.1f} | {err:5.2f} {status}")

if np.all(diff < 0.1):
    print(f"\n✓ ADD 测试通过！所有 {len(expected_add)} 个元素匹配")
else:
    match_count = np.sum(diff < 0.1)
    print(f"\n✗ ADD 测试失败：{match_count}/{len(expected_add)} 个元素匹配")
    print(f"  最大误差: {max_diff:.2f}")

### ADD 测试 - 多种 CHW 配置

测试不同数据尺寸下的 ADD 运算，验证硬件对各种配置的支持。

In [ ]:
# ADD 多配置测试
print("=" * 60)
print("ADD 多配置测试")
print("=" * 60)

# 定义测试用例：(C, H, W, 描述)
test_configs = [
    (8, 1, 1, "最小测试 - 1 BRAM word"),
    (16, 1, 1, "双通道 - 2 BRAM words"),
    (8, 2, 2, "2D小图 - 4 BRAM words"),
    (32, 2, 2, "2D中图 - 16 BRAM words"),
    (128, 4, 4, "大尺寸 - 256 BRAM words"),
]

UNIT_ADD = 6

# 运行每个测试用例
for test_idx, (C, H, W, desc) in enumerate(test_configs):
    print(f"\n{'='*60}")
    print(f"测试用例 {test_idx+1}/{len(test_configs)}: {desc}")
    print(f"配置: C={C}, H={H}, W={W}")
    print(f"{'='*60}")
    
    # 计算数据大小
    SIZE = C * H * W
    BYTES = SIZE * 4
    
    # VPU GB 内地址（避免重叠）
    SRC1_ADDR = 0x0000
    SRC2_ADDR = 0x4000  # 16KB 后
    DST_ADDR = 0x8000   # 32KB 后
    
    # global_bram 地址
    SRC1_OFF = 0x0
    SRC2_OFF = 0x10000  # 64KB 后
    DST_OFF = 0x20000   # 128KB 后
    
    print(f"数据大小: {SIZE} FP32 ({BYTES} bytes = {BYTES//32} BRAM words)")
    
    # 生成测试数据（使用不同范围避免与其他测试混淆）
    base1 = test_idx * 1000 + 1
    base2 = test_idx * 1000 + 10
    src1 = np.arange(base1, base1 + SIZE, dtype=np.float32)
    src2 = np.arange(base2, base2 + SIZE, dtype=np.float32)
    expected = src1 + src2
    
    print(f"SRC1 范围: [{src1.min():.1f}, {src1.max():.1f}]")
    print(f"SRC2 范围: [{src2.min():.1f}, {src2.max():.1f}]")
    print(f"期望结果范围: [{expected.min():.1f}, {expected.max():.1f}]")
    
    # 写入数据
    write_blob(GLOBAL_BRAM_BASE + SRC1_OFF, src1.tobytes())
    write_blob(GLOBAL_BRAM_BASE + SRC2_OFF, src2.tobytes())
    
    # 清零 DST
    dst_zeros = np.zeros(SIZE, dtype=np.float32)
    write_blob(VPU_GB_BASE + DST_ADDR, dst_zeros.tobytes())
    write_blob(GLOBAL_BRAM_BASE + DST_OFF, dst_zeros.tobytes())
    
    # 构建指令序列
    instructions = [
        # CDMA 搬入 SRC1
        encode_cdma_copy(GLOBAL_BRAM_BASE + SRC1_OFF, VPU_GB_BASE + SRC1_ADDR, BYTES),
        encode_wait_cdma(),
        
        # CDMA 搬入 SRC2
        encode_cdma_copy(GLOBAL_BRAM_BASE + SRC2_OFF, VPU_GB_BASE + SRC2_ADDR, BYTES),
        encode_wait_cdma(),
        
        # VPU ADD
        encode_vpu_exec(
            unit_choose=UNIT_ADD,
            src_addr=SRC1_ADDR,
            src2_addr=SRC2_ADDR,
            src_c=C,
            src_h=H,
            src_w=W,
            bias_addr=0,
            scale_addr=0,
            dst_addr=DST_ADDR,
            addr_break=0,
            addr_s=0,
            addr_t=0
        ),
        encode_wait_vpu(),
        
        # CDMA 搬出结果
        encode_cdma_copy(VPU_GB_BASE + DST_ADDR, GLOBAL_BRAM_BASE + DST_OFF, BYTES),
        encode_wait_cdma(),
        
        encode_end(),
    ]
    
    inst_bytes = build_instruction_sequence(instructions)
    inst_count = len(inst_bytes) // 4
    
    # 执行
    write_blob(INST_BRAM_BASE, inst_bytes)
    start_time = time.time()
    decoder_start(inst_count)
    status = decoder_wait(timeout=10.0)
    elapsed = time.time() - start_time
    
    print(f"✓ 执行完成: status=0x{status:08X}, 耗时 {elapsed:.3f}s")
    
    # 读取并验证结果
    result_bytes = read_blob(GLOBAL_BRAM_BASE + DST_OFF, BYTES)
    result = np.frombuffer(result_bytes, dtype=np.float32)
    
    # 数值比较
    diff = np.abs(result - expected)
    max_diff = np.max(diff)
    match_count = np.sum(diff < 0.1)
    match_rate = 100.0 * match_count / SIZE
    
    print(f"\n验证结果:")
    print(f"  最大误差: {max_diff:.4f}")
    print(f"  匹配率: {match_rate:.2f}% ({match_count}/{SIZE})")
    
    if match_rate > 99.9:
        print(f"  ✓ 测试通过!")
    elif match_rate > 90:
        print(f"  ⚠ 部分通过，有少量误差")
        # 显示前几个不匹配的
        mismatch_idx = np.where(diff >= 0.1)[0]
        print(f"  不匹配位置示例 (前5个):")
        for i in range(min(5, len(mismatch_idx))):
            idx = mismatch_idx[i]
            print(f"    [{idx}] 期望={expected[idx]:.2f}, 实际={result[idx]:.2f}, 误差={diff[idx]:.2f}")
    else:
        print(f"  ✗ 测试失败")
        # 显示前10个不匹配的
        mismatch_idx = np.where(diff >= 0.1)[0]
        print(f"  不匹配位置示例 (前10个):")
        for i in range(min(10, len(mismatch_idx))):
            idx = mismatch_idx[i]
            print(f"    [{idx}] 期望={expected[idx]:.2f}, 实际={result[idx]:.2f}, 误差={diff[idx]:.2f}")

print(f"\n{'='*60}")
print(f"ADD 多配置测试完成")
print(f"{'='*60}")

### ADD 测试问题诊断

如果ADD测试失败，可能的原因：

1. **VPU ADD单元未正确实现** - 需要检查RTL代码中的ADD_Unit模块
2. **地址参数问题** - src_addr 和 src2_addr 可能需要不同的配置
3. **数据布局** - VPU可能期望特定的内存布局
4. **UNIT_CHOOSE值** - ADD对应的单元代码可能不是0

建议调试步骤：
- 检查VPU GB中的数据是否正确写入（上面已验证）
- 查看RTL仿真确认ADD单元是否工作
- 尝试不同的UNIT_CHOOSE值（1, 2, 3等）

## 步骤 7: VPU Upsample (×2) 运算测试

US 模块将输入按最近邻插值放大 2 倍。每个输入像素复制到 2×2 的输出块。

- unit_choose = 5 (UNIT_US)
- 参数：src_addr, src_c, src_h, src_w, dst_addr
- 输入: [C, H, W] → 输出: [C, 2H, 2W]

**注意**: 必须在 CDMA 搬运和 VPU 执行之间加入 NOP 延迟，
因为 AXI BRAM Controller 的 Port A 在 CDMA 完成后仍有残留活动，
会干扰 VPU 通过 Port B 读取 BRAM。

In [ ]:
# 测试 7: VPU Upsample ×2 完整流程
import numpy as np
import time

# Upsample 参数
US_C, US_H, US_W = 128, 10, 10
US_OUT_H, US_OUT_W = US_H * 2, US_W * 2  # 20, 20
US_IN_BYTES = US_C * US_H * US_W * 4
US_OUT_BYTES = US_C * US_OUT_H * US_OUT_W * 4

US_SRC = 0x0000
US_DST_OFFSET = US_IN_BYTES  # 输出紧接输入之后

NOP_DELAY = 200  # CDMA 后延迟

print("=" * 60)
print("测试 7: VPU Upsample ×2")
print("=" * 60)
print(f"  输入: C={US_C}, H={US_H}, W={US_W}")
print(f"  输出: C={US_C}, H={US_OUT_H}, W={US_OUT_W}")
print(f"  SRC_ADDR: 0x{US_SRC:04X}")
print(f"  DST_ADDR: 0x{US_DST_OFFSET:04X}")
print(f"  NOP 延迟: {NOP_DELAY}")

# 生成输入数据 (HWC 格式)
us_input = np.arange(US_C * US_H * US_W, dtype=np.float32).reshape(US_H, US_W, US_C)
us_input = us_input / 100.0  # 缩放到合理范围
print(f"\n  输入数据[0,0,:4] = {us_input[0,0,:4]}")
print(f"  输入数据[1,1,:4] = {us_input[1,1,:4]}")

# 清零 VPU GB
write_blob(VPU_GB_BASE + US_SRC, np.zeros(US_IN_BYTES, dtype=np.uint8).tobytes())
write_blob(VPU_GB_BASE + US_DST_OFFSET, np.zeros(US_OUT_BYTES, dtype=np.uint8).tobytes())

# 写入 global_bram
write_blob(GLOBAL_BRAM_BASE, us_input.tobytes())

# 构建指令序列
us_insts = [
    encode_cdma_copy(GLOBAL_BRAM_BASE, VPU_GB_BASE + US_SRC, US_IN_BYTES),
    encode_wait_cdma(),
] + [encode_nop() for _ in range(NOP_DELAY)] + [
    # unit_choose=5 (US), src_addr, src2_addr=0, src_c, src_h, src_w,
    # bias_addr=0, scale_addr=0, dst_addr, addr_break=0, addr_s=0, addr_t=0
    encode_vpu_exec(5, US_SRC, 0, US_C, US_H, US_W, 0, 0, US_DST_OFFSET, 0, 0, 0),
    encode_wait_vpu(),
    encode_cdma_copy(VPU_GB_BASE + US_DST_OFFSET, GLOBAL_BRAM_BASE, US_OUT_BYTES),
    encode_wait_cdma(),
    encode_end()
]

inst_data = build_instruction_sequence(us_insts)
print(f"\n  指令序列大小: {len(inst_data)} 字节 ({len(inst_data)//4} words)")

# 执行
write_blob(INST_BRAM_BASE, inst_data)
decoder_start(len(inst_data) // 4)
time.sleep(4)

# 读取结果
us_output = np.frombuffer(read_blob(GLOBAL_BRAM_BASE, US_OUT_BYTES), dtype=np.float32)
us_output = us_output.reshape(US_OUT_H, US_OUT_W, US_C)

# 计算 Golden Model (Nearest Neighbor ×2)
us_golden = np.zeros((US_OUT_H, US_OUT_W, US_C), dtype=np.float32)
for oh in range(US_OUT_H):
    for ow in range(US_OUT_W):
        us_golden[oh, ow, :] = us_input[oh // 2, ow // 2, :]

# 比较
us_diff = np.abs(us_output - us_golden)
us_match = np.allclose(us_output, us_golden)
print(f"\n  结果比较:")
print(f"    output[0,0,:4] = {us_output[0,0,:4]} (期望 {us_golden[0,0,:4]})")
print(f"    output[0,1,:4] = {us_output[0,1,:4]} (期望 {us_golden[0,1,:4]})")
print(f"    output[1,0,:4] = {us_output[1,0,:4]} (期望 {us_golden[1,0,:4]})")
print(f"    output[1,1,:4] = {us_output[1,1,:4]} (期望 {us_golden[1,1,:4]})")
print(f"    output[2,2,:4] = {us_output[2,2,:4]} (期望 {us_golden[2,2,:4]})")
print(f"    最大误差: {us_diff.max()}")
print(f"    匹配率: {np.sum(us_diff < 1e-6) / us_output.size * 100:.2f}%")
if us_match:
    print("\n  ✓ Upsample ×2 测试通过!")
else:
    print(f"\n  ✗ Upsample ×2 测试失败 (误差元素: {np.sum(us_diff >= 1e-6)})")

## 步骤 8: VPU Quantize (QA) 运算测试

QA 模块将 FP32 输入量化为 INT8：`output = clamp(round(input * scale), -128, 127)`

- unit_choose = 3 (UNIT_QA)
- 参数：src_addr, src_c, src_h, src_w, scale_addr (Weight Buffer 中), dst_addr
- scale 数据存放在 VPU Weight Buffer 中
- 输入: FP32 [C, H, W] → 输出: INT8 [C, H, W] (打包为 256-bit words)

In [ ]:
# 测试 8: VPU Quantize (QA) 功能
import numpy as np
import time

# QA 参数
QA_C, QA_H, QA_W = 8, 1, 1  # 最小测试: 8 个 FP32 = 1 个 BRAM word
QA_IN_BYTES = QA_C * QA_H * QA_W * 4  # FP32 输入
# INT8 输出: 每个 BRAM word 可以存 32 个 INT8
QA_OUT_ELEMENTS = QA_C * QA_H * QA_W
QA_OUT_BYTES = ((QA_OUT_ELEMENTS + 31) // 32) * 32  # 按 32 字节对齐

QA_SRC = 0x0000
QA_DST = 0x1000  # 输出区域
QA_SCALE_ADDR = 0x0000  # scale 在 Weight Buffer 中的地址

NOP_DELAY = 200

print("=" * 60)
print("测试 8: VPU Quantize (QA)")
print("=" * 60)
print(f"  输入: C={QA_C}, H={QA_H}, W={QA_W} (FP32)")
print(f"  输出: INT8")
print(f"  SRC_ADDR: 0x{QA_SRC:04X}, DST_ADDR: 0x{QA_DST:04X}")
print(f"  SCALE_ADDR (WB): 0x{QA_SCALE_ADDR:04X}")

# 生成输入数据
qa_input = np.array([1.0, -2.0, 3.5, -4.5, 5.0, -6.0, 7.5, -8.0], dtype=np.float32)
# Scale = 10.0 (一个 per-tensor scale)
qa_scale = np.array([10.0] * 8, dtype=np.float32)  # 每个 lane 一个 scale

print(f"\n  输入: {qa_input}")
print(f"  Scale: {qa_scale[0]}")
print(f"  期望输出 (round(input*scale)): {np.clip(np.round(qa_input * qa_scale[0]), -128, 127).astype(np.int8)}")

# 写入 Scale 到 Weight Buffer
write_blob(VPU_WB_BASE + QA_SCALE_ADDR, qa_scale.tobytes())

# 写入输入数据到 global_bram
write_blob(GLOBAL_BRAM_BASE, qa_input.tobytes())

# 清零 VPU GB
write_blob(VPU_GB_BASE + QA_SRC, np.zeros(max(QA_IN_BYTES, 32), dtype=np.uint8).tobytes())
write_blob(VPU_GB_BASE + QA_DST, np.zeros(max(QA_OUT_BYTES, 32), dtype=np.uint8).tobytes())

# 构建指令序列
qa_insts = [
    encode_cdma_copy(GLOBAL_BRAM_BASE, VPU_GB_BASE + QA_SRC, QA_IN_BYTES),
    encode_wait_cdma(),
] + [encode_nop() for _ in range(NOP_DELAY)] + [
    # unit_choose=3 (QA), src_addr, src2_addr=0, src_c, src_h, src_w,
    # bias_addr=0, scale_addr (WB地址), dst_addr
    encode_vpu_exec(3, QA_SRC, 0, QA_C, QA_H, QA_W, 0, QA_SCALE_ADDR, QA_DST, 0, 0, 0),
    encode_wait_vpu(),
    encode_cdma_copy(VPU_GB_BASE + QA_DST, GLOBAL_BRAM_BASE, 32),  # 读取 1 个 word
    encode_wait_cdma(),
    encode_end()
]

inst_data = build_instruction_sequence(qa_insts)
print(f"\n  指令序列大小: {len(inst_data)} 字节")

# 执行
write_blob(INST_BRAM_BASE, inst_data)
decoder_start(len(inst_data) // 4)
time.sleep(3)

# 读取结果 (INT8, 打包在 BRAM word 中)
qa_result_raw = read_blob(GLOBAL_BRAM_BASE, 32)
qa_result = np.frombuffer(qa_result_raw, dtype=np.int8)

print(f"\n  输出 (INT8, 前 8 个): {qa_result[:8]}")

# Golden model
qa_golden = np.clip(np.round(qa_input * qa_scale[0]), -128, 127).astype(np.int8)
print(f"  期望 (INT8): {qa_golden}")

if np.array_equal(qa_result[:8], qa_golden):
    print("\n  ✓ QA 测试通过!")
else:
    print(f"\n  ✗ QA 测试失败")
    print(f"    差异: {qa_result[:8] - qa_golden}")

## 步骤 9: VPU Dequantize (DQA) 运算测试

DQA 模块将 INT8 输入反量化为 FP32：`output = input * scale + bias`

- unit_choose = 1 (UNIT_DQA)
- 参数：src_addr, src_c, src_h, src_w, scale_addr, bias_addr, dst_addr
- scale/bias 数据存放在 VPU Weight Buffer 中
- 输入: INT8 [C, H, W] → 输出: FP32 [C, H, W]

In [ ]:
# 测试 9: VPU Dequantize (DQA) 功能
import numpy as np
import time

# DQA 参数: INT8 -> FP32, output = input * scale + bias
DQA_C, DQA_H, DQA_W = 8, 1, 1  # 最小测试
# INT8 输入打包: 8 个 INT8 = 8 bytes, 但 BRAM word = 32 bytes
# DQA 的 C_INT_WIDTH_IN = 32 (按 32 位存储每个 int8，高位补零/符号扩展)
DQA_IN_BYTES = DQA_C * DQA_H * DQA_W * 4  # 每个元素占 32 位
DQA_OUT_BYTES = DQA_C * DQA_H * DQA_W * 4  # FP32 输出

DQA_SRC = 0x0000
DQA_DST = 0x1000
DQA_SCALE_ADDR = 0x0000  # scale 在 WB 中的字节偏移
DQA_BIAS_ADDR = 0x0100   # bias 在 WB 中的字节偏移

NOP_DELAY = 200

print("=" * 60)
print("测试 9: VPU Dequantize (DQA)")
print("=" * 60)
print(f"  输入: C={DQA_C}, H={DQA_H}, W={DQA_W} (INT8 stored as INT32)")
print(f"  输出: FP32")
print(f"  公式: output = input * scale + bias")

# 生成输入数据 (INT8 值，存储为 INT32)
dqa_input_int8 = np.array([10, -20, 30, -40, 50, -60, 70, -80], dtype=np.int32)
# Scale 和 Bias (FP32)
dqa_scale = np.array([0.5] * 8, dtype=np.float32)
dqa_bias = np.array([1.0] * 8, dtype=np.float32)

print(f"\n  输入 (INT8): {dqa_input_int8}")
print(f"  Scale: {dqa_scale[0]}")
print(f"  Bias: {dqa_bias[0]}")

# Golden model
dqa_golden = dqa_input_int8.astype(np.float32) * dqa_scale + dqa_bias
print(f"  期望输出 (FP32): {dqa_golden}")

# 写入 Scale 和 Bias 到 Weight Buffer
write_blob(VPU_WB_BASE + DQA_SCALE_ADDR, dqa_scale.tobytes())
write_blob(VPU_WB_BASE + DQA_BIAS_ADDR, dqa_bias.tobytes())

# 写入输入数据到 global_bram (INT32 格式)
write_blob(GLOBAL_BRAM_BASE, dqa_input_int8.tobytes())

# 清零 VPU GB
write_blob(VPU_GB_BASE + DQA_SRC, np.zeros(max(DQA_IN_BYTES, 32), dtype=np.uint8).tobytes())
write_blob(VPU_GB_BASE + DQA_DST, np.zeros(max(DQA_OUT_BYTES, 32), dtype=np.uint8).tobytes())

# 构建指令序列
# encode_vpu_exec(unit_choose, src_addr, src2_addr, src_c, src_h, src_w,
#                 bias_addr, scale_addr, dst_addr, addr_break, addr_s, addr_t)
dqa_insts = [
    encode_cdma_copy(GLOBAL_BRAM_BASE, VPU_GB_BASE + DQA_SRC, DQA_IN_BYTES),
    encode_wait_cdma(),
] + [encode_nop() for _ in range(NOP_DELAY)] + [
    encode_vpu_exec(1, DQA_SRC, 0, DQA_C, DQA_H, DQA_W, 
                    DQA_BIAS_ADDR, DQA_SCALE_ADDR, DQA_DST, 0, 0, 0),
    encode_wait_vpu(),
    encode_cdma_copy(VPU_GB_BASE + DQA_DST, GLOBAL_BRAM_BASE, DQA_OUT_BYTES),
    encode_wait_cdma(),
    encode_end()
]

inst_data = build_instruction_sequence(dqa_insts)
print(f"\n  指令序列大小: {len(inst_data)} 字节")

# 执行
write_blob(INST_BRAM_BASE, inst_data)
decoder_start(len(inst_data) // 4)
time.sleep(3)

# 读取结果 (FP32)
dqa_result = np.frombuffer(read_blob(GLOBAL_BRAM_BASE, DQA_OUT_BYTES), dtype=np.float32)

print(f"\n  输出 (FP32): {dqa_result}")
print(f"  期望 (FP32): {dqa_golden}")

if np.allclose(dqa_result, dqa_golden, rtol=1e-5, atol=1e-5):
    print("\n  ✓ DQA 测试通过!")
else:
    diff = np.abs(dqa_result - dqa_golden)
    print(f"\n  ✗ DQA 测试失败")
    print(f"    最大误差: {diff.max()}")
    print(f"    逐元素差异: {dqa_result - dqa_golden}")

## 步骤 10: Max Pooling 修正测试 (带 NOP 延迟)

**根因分析结果**: MP RTL 仿真通过，问题出在系统集成层面。

CDMA 通过 AXI BRAM Controller (Port A) 向 VPU GB 写入数据后，
即使 INST_Decoder 收到了 CDMA done 信号，AXI 总线上可能仍有未完成的事务。
此时 MP 单元通过 Port B 读取 BRAM 时，由于 Port A 的残留活动产生干扰，
导致前几个 channel block 的读取结果为 0。

**验证**: 在 CDMA 和 VPU 之间加入 200 个 NOP (约 800 ns @250MHz) 后问题消失。

**永久修复方向**: 
1. 在 INST_Decoder 的 WAIT_CDMA 逻辑中增加额外等待周期
2. 或在 AXI BRAM Controller 配置中增加读写隔离

In [ ]:
# 测试 10: Max Pooling 带 NOP 延迟 (完整验证)
import numpy as np
import time

ACT_C, ACT_H, ACT_W = 128, 10, 10
ACT_BYTES = ACT_C * ACT_H * ACT_W * 4
MP_SRC, MP_DST = 0x0000, 0xC800
NOP_DELAY = 200

print("=" * 60)
print("测试 10: Max Pooling 5×5 (带 NOP 延迟)")
print("=" * 60)

# 生成递增数据 (CHW -> HWC)
data_chw = np.arange(ACT_C * ACT_H * ACT_W, dtype=np.float32).reshape(ACT_C, ACT_H, ACT_W)
data_hwc = np.transpose(data_chw, (1, 2, 0)).copy()

# 准备
write_blob(VPU_GB_BASE + MP_SRC, np.zeros(ACT_BYTES, dtype=np.uint8).tobytes())
write_blob(VPU_GB_BASE + MP_DST, np.zeros(ACT_BYTES, dtype=np.uint8).tobytes())
write_blob(GLOBAL_BRAM_BASE, data_hwc.tobytes())

# 指令序列（带 NOP 延迟）
mp_insts = [
    encode_cdma_copy(GLOBAL_BRAM_BASE, VPU_GB_BASE + MP_SRC, ACT_BYTES),
    encode_wait_cdma(),
] + [encode_nop() for _ in range(NOP_DELAY)] + [
    encode_vpu_exec(4, MP_SRC, 0, ACT_C, ACT_H, ACT_W, 0, 0, MP_DST, 0, 0, 0),
    encode_wait_vpu(),
    encode_cdma_copy(VPU_GB_BASE + MP_DST, GLOBAL_BRAM_BASE, ACT_BYTES),
    encode_wait_cdma(),
    encode_end()
]

inst_data = build_instruction_sequence(mp_insts)
write_blob(INST_BRAM_BASE, inst_data)
decoder_start(len(inst_data) // 4)
time.sleep(4)

# 读取结果
result = np.frombuffer(read_blob(GLOBAL_BRAM_BASE, ACT_BYTES), dtype=np.float32).reshape(ACT_H, ACT_W, ACT_C)

# 计算 Golden Model (5×5 MaxPool, stride=1, pad=2)
golden = np.zeros_like(result)
for oh in range(ACT_H):
    for ow in range(ACT_W):
        for c in range(ACT_C):
            max_val = float('-inf')
            for kh in range(5):
                for kw in range(5):
                    ih = oh + kh - 2
                    iw = ow + kw - 2
                    if 0 <= ih < ACT_H and 0 <= iw < ACT_W:
                        max_val = max(max_val, data_hwc[ih, iw, c])
            golden[oh, ow, c] = max_val

# 比较
diff = np.abs(result - golden)
match_rate = np.sum(diff < 1e-6) / result.size * 100
max_err = diff.max()

print(f"\n  结果比较:")
print(f"    result[0,0,0] = {result[0,0,0]} (期望 {golden[0,0,0]})")
print(f"    result[5,5,0] = {result[5,5,0]} (期望 {golden[5,5,0]})")
print(f"    最大误差: {max_err}")
print(f"    匹配率: {match_rate:.2f}%")

if np.allclose(result, golden):
    print("\n  ✓ Max Pooling 测试通过!")
else:
    # 检查哪些位置有误差
    err_positions = np.argwhere(diff > 1e-6)
    print(f"\n  ✗ Max Pooling 测试失败 (误差元素: {len(err_positions)})")
    if len(err_positions) > 0:
        print(f"    前 5 个误差位置: {err_positions[:5].tolist()}")

## 测试总结

至此，所有VPU硬件测试步骤完成：

1. ✓ BRAM 读写测试 - 验证基础存储访问
2. ✓ VPU 寄存器测试 - 验证配置接口
3. ✓ 指令解码器 - 验证指令执行框架
4. ✓ CDMA 搬运 - 验证数据传输通路
5. ✓ VPU 运算 - 验证 Max Pooling 功能

如果所有步骤都通过，说明VPU硬件工作正常！